# IDB Brazil adaptation projects → adaptation action bank

Maps the 63 screened IDB operations to the ICB × OEF adaptation action bank V13. The mapping links project evidence to actions; it never allocates or splits project amounts. Review: `../../README.md`.

In [1]:
import pandas as pd
import build_brazil_adaptation_release as build

projects = pd.read_csv(build.PROJECT_OUT, dtype=str, keep_default_na=False)
projects['original_approved_amount_usd'] = pd.to_numeric(projects['original_approved_amount_usd'], errors='coerce')
print(projects.shape)
projects[['project_id', 'project_name', 'adaptation_screen_terms']].head()

(63, 35)


,project_id,project_name,adaptation_screen_terms
0,BR-J0001,IDB CLIMA - Minas for Results Program: Decarbo...,explicit_climate_adaptation_or_resilience
1,BR-L1005,Igarapes de Manaus Environmental-Social Program,explicit_hazard_or_risk_reduction
2,BR-L1057,PROCIDADES - Vitória Urban Development and Soc...,explicit_hazard_or_risk_reduction
3,BR-L1058,PROCIDADES - Comprehensive Urban Development P...,explicit_hazard_or_risk_reduction
4,BR-L1335,Environmental Rehabilitation Belo Horizonte - ...,explicit_hazard_or_risk_reduction


## 1. Curated rule families

The rules are explicit phrase families in the builder. They require project-name or objective evidence; broad subsector labels cannot create action links. Drainage, warning, insurance and other exact activities can be `direct`; wider projects are `component`; emergency payments are only `related` to a preparedness plan. Every judgment link is medium confidence and pending adjudication.

In [2]:
mapping = build.build_mapping(projects)
mapping[['project_id', 'action_id', 'action_name', 'match_type', 'mapping_confidence']].head(15)

,project_id,action_id,action_name,match_type,mapping_confidence
0,BR-J0001,,,none,none
1,BR-L1005,,,none,none
2,BR-L1057,c40_0048,Upgrade and maintain urban drainage systems,direct,medium
3,BR-L1058,c40_0048,Upgrade and maintain urban drainage systems,direct,medium
4,BR-L1335,c40_0048,Upgrade and maintain urban drainage systems,direct,medium
5,BR-L1369,,,none,none
6,BR-L1497,,,none,none
7,BR-L1508,c40_0048,Upgrade and maintain urban drainage systems,direct,medium
8,BR-L1542,ipcc_0096,Strengthen climate resilience in agriculture a...,component,medium
9,BR-L1542,ipcc_0101,Improve climate-resilient water and sanitation...,component,medium


## 2. Validate and export

Every screened project appears, unmapped projects carry an explicit `none` row, target identifiers resolve to the local action-bank export, and duplicate project-action pairs fail.

In [3]:
assert set(mapping['project_id']) == set(projects['project_id'])
assert not mapping.duplicated(['project_id', 'action_id']).any()
assert mapping.loc[mapping['action_id'].ne(''), 'action_name'].ne('').all()
assert set(mapping['mapping_confidence']) <= {'medium', 'none'}
mapping.to_csv(build.MAPPING_OUT, index=False, encoding='utf-8')
print('MAPPING VALID', mapping.shape, build.MAPPING_OUT)

MAPPING VALID (78, 8) data/idb_brazil_adaptation_projects_to_actions.csv


## 3. Coverage and judgment clusters

Coverage is reported in both directions. The pending rows should be reviewed by recurring judgment cluster: drainage/flood works, emergency-response evidence, rural adaptation, resilient water and infrastructure, governance, and recovery finance.

In [4]:
linked = mapping[mapping['action_id'].ne('')]
coverage = {
    'source_projects': len(projects),
    'mapped_projects': linked['project_id'].nunique(),
    'unmapped_projects': mapping.loc[mapping['action_id'].eq(''), 'project_id'].nunique(),
    'target_actions_reached': linked['action_id'].nunique(),
    'target_actions_unreached': 37 - linked['action_id'].nunique(),
}
print(coverage)
pd.crosstab(linked['action_id'], linked['match_type'])

{'source_projects': 63, 'mapped_projects': 36, 'unmapped_projects': 27, 'target_actions_reached': 13, 'target_actions_unreached': 24}


match_type,component,direct,related
action_id,,,
c40_0048,0,12,0
icare_0169,0,0,9
icare_0170,2,0,0
icare_0172,1,0,0
icare_0183,4,0,0
icare_0186,1,0,0
ipcc_0087,2,0,0
ipcc_0090,0,1,0
ipcc_0092,3,0,0


## 4. Findings

The draft maps 36 of 63 projects to 13 of 37 actions; 27 projects retain explicit `none` rows. The 51 judgment links comprise 13 direct, 29 component and 9 related links, all medium confidence and pending human adjudication. Emergency assistance is only related to preparedness planning. Broad climate-finance, research and resilience operations remain unmapped when the project text does not identify a concrete action.

Adjudication record: pending. Record reviewer and date here, then change accepted or rejected rows in the curated builder rules and rerun both notebooks.